In [18]:
import nfl_data_py as nfl
import pandas as pd
from pathlib import Path
from datetime import datetime

# Create PickleFiles directory if it doesn't exist
Path("PickleFiles").mkdir(exist_ok=True)

CURRENT_YEAR = 2024  # The most recent completed season

# Load rosters for current year + past years (up to 10 years back)
years = list(range(CURRENT_YEAR - 9, CURRENT_YEAR + 1))
all_rosters = nfl.import_seasonal_rosters(years)

# Filter to fantasy-relevant positions and active players
fantasy_pos = ["QB", "RB", "WR", "TE"]
all_rosters = all_rosters[all_rosters["position"].isin(fantasy_pos)].copy()

# --- Build currYearRoster (current season roster) ---
curr = all_rosters[all_rosters["season"] == CURRENT_YEAR].copy()
curr = curr[curr["status"] == "ACT"]

# Map columns to what PredictionCode expects
curr_roster = pd.DataFrame({
    "Player": curr["player_name"].values,
    "Team": curr["team"].values,
    "Pos": curr["position"].values,
    "Yrs": curr["years_exp"].apply(lambda x: "Rook" if x == 0 else str(int(x))).values,
    "Age": curr["age"].values,
    "BirthDate": curr["birth_date"].values,
})
curr_roster = curr_roster.drop_duplicates(subset=["Player", "BirthDate"]).reset_index(drop=True)
curr_roster.to_pickle("PickleFiles/currYearRoster.pkl")
print(f"currYearRoster: {len(curr_roster)} players")
print(curr_roster.head())

# --- Build pastTeamsRoster (historical rosters with YearsBack) ---
past = all_rosters.copy()
past["YearsBack"] = CURRENT_YEAR - past["season"]

past_roster = pd.DataFrame({
    "Player": past["player_name"].values,
    "Team": past["team"].values,
    "Pos": past["position"].values,
    "BirthDate": past["birth_date"].values,
    "No.": past["jersey_number"].values,
    "YearsBack": past["YearsBack"].values,
})
past_roster = past_roster.drop_duplicates(subset=["Player", "BirthDate", "YearsBack", "Team"]).reset_index(drop=True)
past_roster.to_pickle("PickleFiles/teamsPastRoster.pkl")
print(f"\npastTeamsRoster: {len(past_roster)} rows")
print(f"YearsBack range: {past_roster['YearsBack'].min()} to {past_roster['YearsBack'].max()}")
print(past_roster.head())

currYearRoster: 449 players
             Player Team Pos Yrs   Age  BirthDate
0     Aaron Rodgers  NYJ  QB  19  40.0 1983-12-02
1    Marcedes Lewis  CHI  TE  18  40.0 1984-05-19
2        Joe Flacco  IND  QB  16  39.0 1985-01-16
3      Josh Johnson  BAL  QB  16  38.0 1986-05-15
4  Matthew Stafford   LA  QB  15  36.0 1988-02-07

pastTeamsRoster: 9492 rows
YearsBack range: 0 to 9
            Player Team Pos  BirthDate No.  YearsBack
0  Matt Hasselbeck  IND  QB 1975-09-25   8          9
1   Peyton Manning  DEN  QB 1976-03-24  18          9
2        Tom Brady   NE  QB 1977-08-03  12          9
3     Michael Vick  PIT  QB 1980-06-26   2          9
4      Steve Smith  BLT  WR 1979-05-12  89          9


In [19]:
import nfl_data_py as nfl
import pandas as pd
import numpy as np
from unidecode import unidecode

CURRENT_YEAR = 2024
PEAK_AGE_QB = 30

# Load advanced features ONCE outside dfMaker — covers both YearsBack=1 (2023) and YearsBack=2 (2022)
print("Fetching QB advanced features (passing_epa, pacr, dakota) for 2022-2023...")
qb_adv_raw = nfl.import_seasonal_data([2022, 2023])
qb_adv = qb_adv_raw[['player_id', 'season', 'passing_epa', 'pacr', 'dakota']].copy()
qb_adv['season'] = qb_adv['season'].astype(int)

QB_COLS = ["team", "position", "penalty", "GP", "player_display_name", "age",
           'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions',
           'sacks', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch',
           'passing_first_downs', 'passing_2pt_conversions',
           'carries', 'rushing_yards', 'rushing_tds',
           'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_2pt_conversions',
           'passing_epa', 'pacr', 'dakota',
           "PPG", 'ppg_prev', 'ppg_last_year', 'delta_ppg',
           'age_from_peak', 'age_squared', 'games_missed']

def dfMaker():
    for ppr in range(3):
        # Read files fresh each ppr iteration
        oldQBStats = pd.read_pickle("PickleFiles/final_qb_data.pkl")
        oldQBStats['YearsBack'] = CURRENT_YEAR - oldQBStats['season'].astype(int)
        oldQBStats['season'] = oldQBStats['season'].astype(int)
        # Merge advanced features by player_id + season
        oldQBStats = oldQBStats.merge(qb_adv, on=['player_id', 'season'], how='left')

        currTeamsRoster = pd.read_pickle("PickleFiles/currYearRoster.pkl")
        currAVs = pd.read_pickle("PickleFiles/currAVs.pkl")

        completeDFQB = pd.DataFrame(columns=QB_COLS)
        rookieList = []

        for index in range(len(currTeamsRoster)):
            year = currTeamsRoster.loc[index, 'Yrs']
            bday = currTeamsRoster.loc[index, 'BirthDate']
            pos  = currTeamsRoster.loc[index, 'Pos']
            age  = currTeamsRoster.loc[index, 'Age']
            name = currTeamsRoster.loc[index, 'Player']
            team = currTeamsRoster.loc[index, 'Team']

            if pos != "QB":
                continue

            # Skip rookies
            if year == "Rook":
                rookieList.append({"Name": name, "Year": year, "Bday": bday, "Age": age, "Team": team})
                continue

            # Single-season lookup: prefer YearsBack=1 if GP>=8, else fall back to YearsBack=2
            r1 = oldQBStats[(oldQBStats['player_display_name'] == name) & (oldQBStats['YearsBack'] == 1)].copy()
            r2 = oldQBStats[(oldQBStats['player_display_name'] == name) & (oldQBStats['YearsBack'] == 2)].copy()

            if not r1.empty and r1.iloc[0]['GP'] >= 8:
                row = r1.iloc[0]
            elif not r2.empty:
                row = r2.iloc[0]
            else:
                continue  # no usable data

            gp = float(row['GP'])
            if gp < 6:
                continue

            # Build per-game counting stats
            count_cols = ['completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions',
                          'sacks', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch',
                          'passing_first_downs', 'passing_2pt_conversions',
                          'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles_lost',
                          'rushing_first_downs', 'rushing_2pt_conversions']

            per_game = {col: float(row.get(col, 0)) / gp for col in count_cols}

            # Advanced features
            passing_epa = float(row['passing_epa']) / gp if pd.notna(row.get('passing_epa')) else 0.0
            pacr         = float(row['pacr'])           if pd.notna(row.get('pacr'))         else 0.0
            dakota       = float(row['dakota'])         if pd.notna(row.get('dakota'))        else 0.0

            # PPG from per-game stats
            PPG = (per_game['rushing_yards'] * 0.1 +
                   per_game['passing_yards'] * 0.04 +
                   per_game['rushing_tds'] * 6 +
                   per_game['passing_tds'] * 4 +
                   per_game['rushing_fumbles_lost'] * -2 +
                   per_game['sack_fumbles_lost'] * -2 +
                   per_game['interceptions'] * -2)

            # ppg_prev = YearsBack=1 PPG (always from r1, regardless of which row was chosen for stats)
            if not r1.empty:
                r1row = r1.iloc[0]
                gp1 = max(float(r1row['GP']), 1)
                fp1 = float(r1row.get('fantasy_points', 0))
                ppg_yb1 = fp1 / gp1
            else:
                ppg_yb1 = PPG

            if not r2.empty:
                r2row = r2.iloc[0]
                gp2 = max(float(r2row['GP']), 1)
                fp2 = float(r2row.get('fantasy_points', 0))
                ppg_yb2 = fp2 / gp2
            else:
                ppg_yb2 = ppg_yb1

            ppg_prev     = ppg_yb1
            ppg_last_year = ppg_yb2
            delta_ppg    = ppg_yb1 - ppg_yb2
            age_from_peak = float(age) - PEAK_AGE_QB
            age_squared  = float(age) ** 2
            games_missed = max(0, 17 - (int(r1.iloc[0]['GP']) if not r1.empty else 17))
            penalty      = gp  # penalty placeholder (same as original GP-based value)

            row_data = {
                "team": team, "position": pos, "penalty": penalty, "GP": gp,
                "player_display_name": name, "age": float(age),
                **per_game,
                "passing_epa": passing_epa, "pacr": pacr, "dakota": dakota,
                "PPG": PPG,
                "ppg_prev": ppg_prev, "ppg_last_year": ppg_last_year, "delta_ppg": delta_ppg,
                "age_from_peak": age_from_peak, "age_squared": age_squared,
                "games_missed": games_missed
            }

            individualDFQB = pd.DataFrame([row_data], columns=QB_COLS)
            completeDFQB = pd.concat([completeDFQB, individualDFQB], ignore_index=True)

        # Save
        completeDFQB = pd.merge(completeDFQB, currAVs, on='team', how='left')
        completeDFQB = completeDFQB.fillna(0)
        completeDFQB = completeDFQB.sort_values(by='PPG')
        pkl_map = {0: "PickleFiles/QBDFForModelNonPPR.pkl",
                   1: "PickleFiles/QBDFForModelHalfPPR.pkl",
                   2: "PickleFiles/QBDFForModelPPR.pkl"}
        completeDFQB.to_pickle(pkl_map[ppr])
        print(f"  QB ppr={ppr}: {len(completeDFQB)} players saved -> {pkl_map[ppr]}")

dfMaker()


Fetching QB advanced features (passing_epa, pacr, dakota) for 2022-2023...


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/863479526.py:131: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  completeDFQB = pd.concat([completeDFQB, individualDFQB], ignore_index=True)
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/863479526.py:135: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  completeDFQB = completeDFQB.fillna(0)
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/863479526.py:131: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA en

  QB ppr=0: 41 players saved -> PickleFiles/QBDFForModelNonPPR.pkl
  QB ppr=1: 41 players saved -> PickleFiles/QBDFForModelHalfPPR.pkl
  QB ppr=2: 41 players saved -> PickleFiles/QBDFForModelPPR.pkl


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/863479526.py:135: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  completeDFQB = completeDFQB.fillna(0)
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/863479526.py:131: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  completeDFQB = pd.concat([completeDFQB, individualDFQB], ignore_index=True)
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/863479526.py:135: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is d

In [20]:
import nfl_data_py as nfl
import pandas as pd
import numpy as np
from unidecode import unidecode

CURRENT_YEAR = 2024
PEAK_AGE_RB = 25

# Load opportunity + EPA features ONCE outside dfMaker
print("Fetching RB opportunity/EPA features for 2022-2023...")
rb_opp_raw = nfl.import_seasonal_data([2022, 2023])
rb_opp = rb_opp_raw[['player_id', 'season',
                       'target_share', 'ry_sh',
                       'rushing_epa', 'receiving_epa',
                       'rtd_sh', 'rfd_sh']].copy()
rb_opp['season'] = rb_opp['season'].astype(int)

RB_COLS = ["team", "position", "penalty", "GP", "player_display_name", "age",
           'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles_lost',
           'rushing_first_downs', 'rushing_2pt_conversions',
           'receptions', 'targets', 'receiving_yards', 'receiving_tds',
           'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch',
           'receiving_first_downs', 'receiving_2pt_conversions', 'special_teams_tds', 'rrtd',
           "PPG",
           'target_share', 'ry_sh', 'rushing_epa', 'receiving_epa', 'rtd_sh', 'rfd_sh',
           'ppg_prev', 'ppg_last_year', 'delta_ppg',
           'age_from_peak', 'age_squared', 'games_missed']

def dfMaker():
    for ppr in range(3):
        oldRBStats = pd.read_pickle("PickleFiles/final_rb_data.pkl")
        oldRBStats['YearsBack'] = CURRENT_YEAR - oldRBStats['season'].astype(int)
        oldRBStats['season'] = oldRBStats['season'].astype(int)
        oldRBStats = oldRBStats.merge(rb_opp, on=['player_id', 'season'], how='left')

        currTeamsRoster = pd.read_pickle("PickleFiles/currYearRoster.pkl")
        currAVs = pd.read_pickle("PickleFiles/currAVs.pkl")

        completeDFRB = pd.DataFrame(columns=RB_COLS)
        rookieList = []

        for index in range(len(currTeamsRoster)):
            year = currTeamsRoster.loc[index, 'Yrs']
            bday = currTeamsRoster.loc[index, 'BirthDate']
            pos  = currTeamsRoster.loc[index, 'Pos']
            age  = currTeamsRoster.loc[index, 'Age']
            name = currTeamsRoster.loc[index, 'Player']
            team = currTeamsRoster.loc[index, 'Team']

            if pos != "RB":
                continue

            if year == "Rook":
                rookieList.append({"Name": name, "Year": year, "Bday": bday, "Age": age, "Team": team})
                continue

            r1 = oldRBStats[(oldRBStats['player_display_name'] == name) & (oldRBStats['YearsBack'] == 1)].copy()
            r2 = oldRBStats[(oldRBStats['player_display_name'] == name) & (oldRBStats['YearsBack'] == 2)].copy()

            if not r1.empty and r1.iloc[0]['GP'] >= 8:
                row = r1.iloc[0]
            elif not r2.empty:
                row = r2.iloc[0]
            else:
                continue

            gp = float(row['GP'])
            if gp < 6:
                continue

            count_cols = ['carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles_lost',
                          'rushing_first_downs', 'rushing_2pt_conversions',
                          'receptions', 'targets', 'receiving_yards', 'receiving_tds',
                          'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch',
                          'receiving_first_downs', 'receiving_2pt_conversions', 'special_teams_tds', 'rrtd']
            per_game = {col: float(row.get(col, 0)) / gp for col in count_cols}

            # Opportunity: shares/ratios — no divide
            target_share  = float(row['target_share'])  if pd.notna(row.get('target_share'))  else 0.0
            ry_sh         = float(row['ry_sh'])         if pd.notna(row.get('ry_sh'))         else 0.0
            rtd_sh        = float(row['rtd_sh'])        if pd.notna(row.get('rtd_sh'))        else 0.0
            rfd_sh        = float(row['rfd_sh'])        if pd.notna(row.get('rfd_sh'))        else 0.0
            # EPA totals -> per game
            rushing_epa   = float(row['rushing_epa'])   / gp if pd.notna(row.get('rushing_epa'))   else 0.0
            receiving_epa = float(row['receiving_epa']) / gp if pd.notna(row.get('receiving_epa')) else 0.0

            # PPG (base, without PPR adjustment yet — add receptions below)
            base_ppg = (per_game['rushing_yards'] * 0.1 +
                        per_game['rushing_tds'] * 6 +
                        per_game['rushing_fumbles_lost'] * -2 +
                        per_game['receiving_yards'] * 0.1 +
                        per_game['receiving_tds'] * 6 +
                        per_game['receiving_fumbles_lost'] * -2)
            rec_pg = per_game['receptions']
            if ppr == 2:
                PPG = base_ppg + rec_pg
            elif ppr == 1:
                PPG = base_ppg + rec_pg * 0.5
            else:
                PPG = base_ppg

            # ppg_prev / ppg_last_year — always from r1 / r2
            def rb_ppg_from_row(r, p):
                gp_ = max(float(r['GP']), 1)
                b = (float(r.get('rushing_yards', 0)) * 0.1 +
                     float(r.get('rushing_tds', 0)) * 6 +
                     float(r.get('rushing_fumbles_lost', 0)) * -2 +
                     float(r.get('receiving_yards', 0)) * 0.1 +
                     float(r.get('receiving_tds', 0)) * 6 +
                     float(r.get('receiving_fumbles_lost', 0)) * -2) / gp_
                rec_ = float(r.get('receptions', 0)) / gp_
                return b + (rec_ if p == 2 else rec_ * 0.5 if p == 1 else 0)

            ppg_yb1 = rb_ppg_from_row(r1.iloc[0], ppr) if not r1.empty else PPG
            ppg_yb2 = rb_ppg_from_row(r2.iloc[0], ppr) if not r2.empty else ppg_yb1

            ppg_prev      = ppg_yb1
            ppg_last_year = ppg_yb2
            delta_ppg     = ppg_yb1 - ppg_yb2
            age_from_peak = float(age) - PEAK_AGE_RB
            age_squared   = float(age) ** 2
            games_missed  = max(0, 17 - int(r1.iloc[0]['GP'])) if not r1.empty else 17
            penalty       = gp

            row_data = {
                "team": team, "position": pos, "penalty": penalty, "GP": gp,
                "player_display_name": name, "age": float(age),
                **per_game,
                "PPG": PPG,
                "target_share": target_share, "ry_sh": ry_sh,
                "rushing_epa": rushing_epa, "receiving_epa": receiving_epa,
                "rtd_sh": rtd_sh, "rfd_sh": rfd_sh,
                "ppg_prev": ppg_prev, "ppg_last_year": ppg_last_year, "delta_ppg": delta_ppg,
                "age_from_peak": age_from_peak, "age_squared": age_squared,
                "games_missed": games_missed
            }

            individualDFRB = pd.DataFrame([row_data], columns=RB_COLS)
            completeDFRB = pd.concat([completeDFRB, individualDFRB], ignore_index=True)

        completeDFRB = pd.merge(completeDFRB, currAVs, on='team', how='left')
        completeDFRB = completeDFRB.fillna(0)
        completeDFRB = completeDFRB.sort_values(by='PPG')
        pkl_map = {0: "PickleFiles/RBDFForModelNonPPR.pkl",
                   1: "PickleFiles/RBDFForModelHalfPPR.pkl",
                   2: "PickleFiles/RBDFForModelPPR.pkl"}
        completeDFRB.to_pickle(pkl_map[ppr])
        print(f"  RB ppr={ppr}: {len(completeDFRB)} players saved -> {pkl_map[ppr]}")

dfMaker()


Fetching RB opportunity/EPA features for 2022-2023...


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/3603843344.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  completeDFRB = pd.concat([completeDFRB, individualDFRB], ignore_index=True)
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/3603843344.py:142: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  completeDFRB = completeDFRB.fillna(0)
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/3603843344.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA

  RB ppr=0: 70 players saved -> PickleFiles/RBDFForModelNonPPR.pkl


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/3603843344.py:142: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  completeDFRB = completeDFRB.fillna(0)
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/3603843344.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  completeDFRB = pd.concat([completeDFRB, individualDFRB], ignore_index=True)


  RB ppr=1: 70 players saved -> PickleFiles/RBDFForModelHalfPPR.pkl
  RB ppr=2: 70 players saved -> PickleFiles/RBDFForModelPPR.pkl


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/3603843344.py:142: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  completeDFRB = completeDFRB.fillna(0)


In [21]:
import nfl_data_py as nfl
import pandas as pd
import numpy as np
from unidecode import unidecode

CURRENT_YEAR = 2024
PEAK_AGE_WRTE = 26

# Load opportunity + EPA features ONCE outside dfMaker
print("Fetching WR/TE opportunity/EPA features for 2022-2023...")
wrte_opp_raw = nfl.import_seasonal_data([2022, 2023])
wrte_opp = wrte_opp_raw[['player_id', 'season',
                           'target_share', 'air_yards_share',
                           'wopr_x', 'racr', 'receiving_epa',
                           'yac_sh', 'tgt_sh']].copy()
wrte_opp['season'] = wrte_opp['season'].astype(int)

WRTE_COLS = ["team", "position", "penalty", "GP", "player_display_name", "age",
             'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles_lost',
             'rushing_first_downs', 'rushing_2pt_conversions',
             'receptions', 'targets', 'receiving_yards', 'receiving_tds',
             'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch',
             'receiving_first_downs', 'receiving_2pt_conversions', 'special_teams_tds', 'rrtd',
             "PPG",
             'target_share', 'air_yards_share', 'wopr_x', 'racr', 'receiving_epa', 'yac_sh', 'tgt_sh',
             'ppg_prev', 'ppg_last_year', 'delta_ppg',
             'age_from_peak', 'age_squared', 'games_missed']

def dfMaker():
    for ppr in range(3):
        oldWRTEStats = pd.read_pickle("PickleFiles/final_wrte_data.pkl")
        oldWRTEStats['YearsBack'] = CURRENT_YEAR - oldWRTEStats['season'].astype(int)
        oldWRTEStats['season'] = oldWRTEStats['season'].astype(int)
        oldWRTEStats = oldWRTEStats.merge(wrte_opp, on=['player_id', 'season'], how='left')

        currTeamsRoster = pd.read_pickle("PickleFiles/currYearRoster.pkl")
        currAVs = pd.read_pickle("PickleFiles/currAVs.pkl")

        completeDFWRTE = pd.DataFrame(columns=WRTE_COLS)
        rookieList = []

        for index in range(len(currTeamsRoster)):
            year = currTeamsRoster.loc[index, 'Yrs']
            bday = currTeamsRoster.loc[index, 'BirthDate']
            pos  = currTeamsRoster.loc[index, 'Pos']
            age  = currTeamsRoster.loc[index, 'Age']
            name = currTeamsRoster.loc[index, 'Player']
            team = currTeamsRoster.loc[index, 'Team']

            if pos not in ("WR", "TE"):
                continue

            if year == "Rook":
                rookieList.append({"Name": name, "Year": year, "Bday": bday, "Age": age, "Team": team})
                continue

            r1 = oldWRTEStats[(oldWRTEStats['player_display_name'] == name) & (oldWRTEStats['YearsBack'] == 1)].copy()
            r2 = oldWRTEStats[(oldWRTEStats['player_display_name'] == name) & (oldWRTEStats['YearsBack'] == 2)].copy()

            if not r1.empty and r1.iloc[0]['GP'] >= 8:
                row = r1.iloc[0]
            elif not r2.empty:
                row = r2.iloc[0]
            else:
                continue

            gp = float(row['GP'])
            if gp < 6:
                continue

            count_cols = ['carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles_lost',
                          'rushing_first_downs', 'rushing_2pt_conversions',
                          'receptions', 'targets', 'receiving_yards', 'receiving_tds',
                          'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch',
                          'receiving_first_downs', 'receiving_2pt_conversions', 'special_teams_tds', 'rrtd']
            per_game = {col: float(row.get(col, 0)) / gp for col in count_cols}

            # Shares/ratios — no divide
            target_share    = float(row['target_share'])    if pd.notna(row.get('target_share'))    else 0.0
            air_yards_share = float(row['air_yards_share']) if pd.notna(row.get('air_yards_share')) else 0.0
            wopr_x          = float(row['wopr_x'])          if pd.notna(row.get('wopr_x'))          else 0.0
            racr            = float(row['racr'])             if pd.notna(row.get('racr'))             else 0.0
            yac_sh          = float(row['yac_sh'])           if pd.notna(row.get('yac_sh'))           else 0.0
            tgt_sh          = float(row['tgt_sh'])           if pd.notna(row.get('tgt_sh'))           else 0.0
            # receiving_epa is season total -> per game
            receiving_epa   = float(row['receiving_epa']) / gp if pd.notna(row.get('receiving_epa')) else 0.0

            base_ppg = (per_game['rushing_yards'] * 0.1 +
                        per_game['rushing_tds'] * 6 +
                        per_game['rushing_fumbles_lost'] * -2 +
                        per_game['receiving_yards'] * 0.1 +
                        per_game['receiving_tds'] * 6 +
                        per_game['receiving_fumbles_lost'] * -2)
            rec_pg = per_game['receptions']
            if ppr == 2:
                PPG = base_ppg + rec_pg
            elif ppr == 1:
                PPG = base_ppg + rec_pg * 0.5
            else:
                PPG = base_ppg

            def wrte_ppg_from_row(r, p):
                gp_ = max(float(r['GP']), 1)
                b = (float(r.get('rushing_yards', 0)) * 0.1 +
                     float(r.get('rushing_tds', 0)) * 6 +
                     float(r.get('rushing_fumbles_lost', 0)) * -2 +
                     float(r.get('receiving_yards', 0)) * 0.1 +
                     float(r.get('receiving_tds', 0)) * 6 +
                     float(r.get('receiving_fumbles_lost', 0)) * -2) / gp_
                rec_ = float(r.get('receptions', 0)) / gp_
                return b + (rec_ if p == 2 else rec_ * 0.5 if p == 1 else 0)

            ppg_yb1 = wrte_ppg_from_row(r1.iloc[0], ppr) if not r1.empty else PPG
            ppg_yb2 = wrte_ppg_from_row(r2.iloc[0], ppr) if not r2.empty else ppg_yb1

            ppg_prev      = ppg_yb1
            ppg_last_year = ppg_yb2
            delta_ppg     = ppg_yb1 - ppg_yb2
            age_from_peak = float(age) - PEAK_AGE_WRTE
            age_squared   = float(age) ** 2
            games_missed  = max(0, 17 - int(r1.iloc[0]['GP'])) if not r1.empty else 17
            penalty       = gp

            row_data = {
                "team": team, "position": pos, "penalty": penalty, "GP": gp,
                "player_display_name": name, "age": float(age),
                **per_game,
                "PPG": PPG,
                "target_share": target_share, "air_yards_share": air_yards_share,
                "wopr_x": wopr_x, "racr": racr, "receiving_epa": receiving_epa,
                "yac_sh": yac_sh, "tgt_sh": tgt_sh,
                "ppg_prev": ppg_prev, "ppg_last_year": ppg_last_year, "delta_ppg": delta_ppg,
                "age_from_peak": age_from_peak, "age_squared": age_squared,
                "games_missed": games_missed
            }

            individualDFWRTE = pd.DataFrame([row_data], columns=WRTE_COLS)
            completeDFWRTE = pd.concat([completeDFWRTE, individualDFWRTE], ignore_index=True)

        completeDFWRTE = pd.merge(completeDFWRTE, currAVs, on='team', how='left')
        completeDFWRTE = completeDFWRTE.fillna(0)
        completeDFWRTE = completeDFWRTE.sort_values(by='PPG')
        pkl_map = {0: "PickleFiles/WRTEDFForModelNonPPR.pkl",
                   1: "PickleFiles/WRTEDFForModelHalfPPR.pkl",
                   2: "PickleFiles/WRTEDFForModelPPR.pkl"}
        completeDFWRTE.to_pickle(pkl_map[ppr])
        print(f"  WRTE ppr={ppr}: {len(completeDFWRTE)} players saved -> {pkl_map[ppr]}")

dfMaker()


Fetching WR/TE opportunity/EPA features for 2022-2023...


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/721253947.py:138: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  completeDFWRTE = pd.concat([completeDFWRTE, individualDFWRTE], ignore_index=True)
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/721253947.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  completeDFWRTE = completeDFWRTE.fillna(0)
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/721253947.py:138: FutureWarning: The behavior of DataFrame concatenation with empty or

  WRTE ppr=0: 174 players saved -> PickleFiles/WRTEDFForModelNonPPR.pkl


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/721253947.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  completeDFWRTE = completeDFWRTE.fillna(0)
/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/721253947.py:138: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  completeDFWRTE = pd.concat([completeDFWRTE, individualDFWRTE], ignore_index=True)


  WRTE ppr=1: 174 players saved -> PickleFiles/WRTEDFForModelHalfPPR.pkl
  WRTE ppr=2: 174 players saved -> PickleFiles/WRTEDFForModelPPR.pkl


/var/folders/12/yvdl_th13tl4mjm1qz4j380m0000gn/T/ipykernel_8283/721253947.py:141: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  completeDFWRTE = completeDFWRTE.fillna(0)


In [22]:
import pandas as pd
import numpy as np
from itertools import chain
import joblib
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings("ignore")

def getScaleBack(df):
    column_index = df.columns.get_loc("PPG")
    min_value = df["PPG"].min()
    max_value = df["PPG"].max()
    return [min_value, max_value]

def scorer():
    scaler = MinMaxScaler()

    for ppr in range(3):
        dictScores = {}

        if ppr == 0:
            qb   = pd.read_pickle("PickleFiles/QBDFForModelNonPPR.pkl")
            rb   = pd.read_pickle("PickleFiles/RBDFForModelNonPPR.pkl")
            wrte = pd.read_pickle("PickleFiles/WRTEDFForModelNonPPR.pkl")

            rbModel   = joblib.load("rb models/rbModelNonPPR.joblib")
            wrModel   = joblib.load("wrte models/wrModelNonPPR.joblib")
            teModel   = joblib.load("wrte models/teModelNonPPR.joblib")
            qbModel   = joblib.load("qb models/qbModelNonPPR.joblib")
        elif ppr == 1:
            qb   = pd.read_pickle("PickleFiles/QBDFForModelHalfPPR.pkl")
            rb   = pd.read_pickle("PickleFiles/RBDFForModelHalfPPR.pkl")
            wrte = pd.read_pickle("PickleFiles/WRTEDFForModelHalfPPR.pkl")

            rbModel   = joblib.load("rb models/rbModelHalfPPR.joblib")
            wrModel   = joblib.load("wrte models/wrModelHalfPPR.joblib")
            teModel   = joblib.load("wrte models/teModelHalfPPR.joblib")
            qbModel   = joblib.load("qb models/qbModelHalfPPR.joblib")
        elif ppr == 2:
            qb   = pd.read_pickle("PickleFiles/QBDFForModelPPR.pkl")
            rb   = pd.read_pickle("PickleFiles/RBDFForModelPPR.pkl")
            wrte = pd.read_pickle("PickleFiles/WRTEDFForModelPPR.pkl")

            rbModel   = joblib.load("rb models/rbModelPPR.joblib")
            wrModel   = joblib.load("wrte models/wrModelPPR.joblib")
            teModel   = joblib.load("wrte models/teModelPPR.joblib")
            qbModel   = joblib.load("qb models/qbModelPPR.joblib")

        # Route by position: WR -> wrModel, TE -> teModel, RB -> rbModel, QB -> qbModel
        modelsDict = {"QB": qbModel, "WR": wrModel, "TE": teModel, "RB": rbModel}

        scaleQB   = getScaleBack(qb)
        scaleRB   = getScaleBack(rb)
        scaleWRTE = getScaleBack(wrte)

        # Scale each positional df (drop metadata columns)
        drop_meta = ["GP", "player_display_name", "team", "position", "penalty", "PPG"]

        rbScaled = rb.copy()
        rbScaled = rbScaled.drop(columns=drop_meta, errors='ignore')
        rbScaled[rbScaled.columns] = scaler.fit_transform(rbScaled[rbScaled.columns])

        wrteScaled = wrte.copy()
        wrteScaled = wrteScaled.drop(columns=drop_meta, errors='ignore')
        wrteScaled[wrteScaled.columns] = scaler.fit_transform(wrteScaled[wrteScaled.columns])

        qbScaled = qb.copy()
        qbScaled = qbScaled.drop(columns=drop_meta, errors='ignore')
        qbScaled[qbScaled.columns] = scaler.fit_transform(qbScaled[qbScaled.columns])

        allPosDFs       = [rb, wrte, qb]
        allPosDfsScaled = [rbScaled, wrteScaled, qbScaled]
        scaleBack       = [scaleRB, scaleWRTE, scaleQB]

        indPosArr = []

        for ind in range(len(allPosDFs)):
            currDF = allPosDFs[ind]
            scaled = allPosDfsScaled[ind]
            arr    = scaleBack[ind]

            currDF["Final PPG"] = 0

            for i in range(len(currDF)):
                currRow = currDF.iloc[[i]]
                currRow = currRow.reset_index()
                scaled  = scaled.reset_index()
                scaled  = scaled.drop(columns=["index"], errors='ignore')
                currRowForModel = scaled.iloc[[i]]

                pos  = currRow.loc[0, "position"]
                name = currRow.loc[0, "player_display_name"]
                team = currRow.loc[0, "team"]

                # Route directly by position (WR and TE are separate models)
                model = modelsDict[pos]
                currRowForModel = currRowForModel[model.feature_names_in_]
                prediction = model.predict(currRowForModel)

                prediction = (prediction * (arr[1] - arr[0])) + arr[0]
                currDF.at[i, "Final PPG"] = prediction[0]

            indPosArr.append(currDF)

        finalrbs  = indPosArr[0].sort_values(by="Final PPG", ascending=False)
        finalwrte = indPosArr[1].sort_values(by="Final PPG", ascending=False)
        finalqbs  = indPosArr[2].sort_values(by="Final PPG", ascending=False)

        finalrbs["Rank"]  = np.arange(1, len(finalrbs) + 1)
        finalwrte["Rank"] = np.arange(1, len(finalwrte) + 1)
        finalqbs["Rank"]  = np.arange(1, len(finalqbs) + 1)

        finalrbs  = finalrbs[["Rank",  "player_display_name", "team", "position", "Final PPG"]]
        finalwrte = finalwrte[["Rank", "player_display_name", "team", "position", "Final PPG"]]
        finalqbs  = finalqbs[["Rank",  "player_display_name", "team", "position", "Final PPG"]]

        finalrbs.columns  = ["Rank", "Name", "Team", "Position", "Final PPG"]
        finalwrte.columns = ["Rank", "Name", "Team", "Position", "Final PPG"]
        finalqbs.columns  = ["Rank", "Name", "Team", "Position", "Final PPG"]

        if ppr == 0:
            finalrbs.to_pickle("PickleFiles/RBs_NonPPR.pkl")
            finalwrte.to_pickle("PickleFiles/WRTE_NonPPR.pkl")
            finalqbs.to_pickle("PickleFiles/QBs_NonPPR.pkl")
        elif ppr == 1:
            finalrbs.to_pickle("PickleFiles/RBs_HalfPPR.pkl")
            finalwrte.to_pickle("PickleFiles/WRTE_HalfPPR.pkl")
            finalqbs.to_pickle("PickleFiles/QBs_HalfPPR.pkl")
        elif ppr == 2:
            finalrbs.to_pickle("PickleFiles/RBs_PPR.pkl")
            finalwrte.to_pickle("PickleFiles/WRTE_PPR.pkl")
            finalqbs.to_pickle("PickleFiles/QBs_PPR.pkl")

scorer()
